In [ ]:
import pandas as pd
import re

# 1. Load the raw data
df_raw = pd.read_csv("ingredient_dish_edges.csv")

# 2. Filter using only your 36 regional cuisines
indian_cuisine = (
    'Andhra', 'Udupi', 'Bengali Recipes', 'Punjabi', 'Chettinad',
    'Tamil Nadu', 'Maharashtrian Recipes', 'Sindhi', 'Kerala Recipes',
    'Gujarati Recipes\ufeff', 'Coorg', 'Rajasthani', 'Kashmiri',
    'Karnataka', 'Lucknowi', 'Hyderabadi', 'Goan Recipes', 'Assamese',
    'Bihari', 'Malabar', 'Himachal', 'Awadhi', 'North East India Recipes',
    'Mangalorean', 'North Karnataka', 'Parsi Recipes', 'Oriya Recipes',
    'Konkan', 'Uttar Pradesh', 'Malvani', 'Kongunadu', 'South Karnataka',
    'Haryana', 'Uttarakhand-North Kumaon', 'Jharkhand', 'Nagaland'
)
df_indian = df_raw[df_raw['Region'].isin(indian_cuisine)].copy()

# Drop the Quantity column
df_indian = df_indian[['Dish_ID', 'Dish', 'Ingredient', 'Region']].copy()

# 3. Clean the ingredients
def clean_ingredient(ing):
    ing = str(ing).lower().strip()
    
    # Remove numbers and measurement words
    ing = re.sub(r"^(?:\+|\d+|to\s+\d+|or\s+\d+|make\s+\d+|cut\s+\d+|tighten\s+\d+|[¼½¾⅓⅔])\s*", "", ing)
    ing = re.sub(r"^(?:\d+//?\d+\s*)", "", ing)
    ing = re.sub(r"^(?:cups?|tablespoons?|tbsp|teaspoons?|tsp|inch|sprigs?\s+of|sprigs?|cloves?|liter|grams?)\s+", "", ing)
    ing = ing.strip()
    
    # Merge duplicates and translation artifacts
    mapping = {
        "long": "cloves", "laung": "cloves", "clove": "cloves",
        "rye": "mustard seeds", "rai": "mustard seeds", "mustard": "mustard seeds",
        "good": "jaggery", "save": "sev", "बड़े चमच्च हींग": "asafoetida",
        "onions": "onion", "tomatoes": "tomato", "potatoes": "potato", "carrots": "carrot", "eggs": "egg",
        "green chili": "green chillies", "green chilli": "green chillies", "green chilies": "green chillies",
        "dry red chilli": "dry red chillies", "dry red chili": "dry red chillies", "dry red chilies": "dry red chillies",
        "red chili powder": "red chilli powder", "kashmiri red chili powder": "kashmiri red chilli powder",
        "kashmiri dry red chilli": "kashmiri dry red chillies", "fresh red chilli": "fresh red chillies",
        "coriander": "coriander leaves", "yogurt": "curd"
    }
    return mapping.get(ing, ing)

df_indian["Ingredient"] = df_indian["Ingredient"].apply(clean_ingredient)

# Remove garbage words and drop duplicate rows
garbage_words = {"", "1/2", "yield", "peel", "mash", "mess", "wood", "hung", "leaf"}
df_indian = df_indian[~df_indian["Ingredient"].isin(garbage_words)]
df_indian = df_indian.drop_duplicates(subset=["Dish_ID", "Ingredient"])

# 4. Generate Nodes Data
recipe_nodes = df_indian[["Dish_ID", "Dish", "Region"]].drop_duplicates(subset="Dish_ID")
recipe_nodes = recipe_nodes.rename(columns={"Dish_ID": "Id", "Dish": "Label", "Region": "Cuisine"})
recipe_nodes["Type"] = "Recipe"

ingredient_nodes = pd.DataFrame({"Id": df_indian["Ingredient"].unique()})
ingredient_nodes["Label"] = ingredient_nodes["Id"]
ingredient_nodes["Cuisine"] = ""
ingredient_nodes["Type"] = "Ingredient"

nodes_clean = pd.concat([recipe_nodes, ingredient_nodes], ignore_index=True)

# 5. Generate Edges Data
edges_clean = df_indian[["Dish_ID", "Ingredient"]].rename(columns={"Dish_ID": "Source", "Ingredient": "Target"})
z
# 6. Export to CSV
nodes_clean.to_csv("nodes_clean.csv", index=False)
edges_clean.to_csv("edges_clean.csv", index=False)

print(f"Success! Exported nodes_clean.csv ({len(nodes_clean)} nodes) and edges_clean.csv ({len(edges_clean)} edges).")

Success! Exported nodes_clean.csv (2502 nodes) and edges_clean.csv (20699 edges).
